# ResNet

In [22]:
from datasets import load_from_disk
loaded_dataset = load_from_disk('./dataset')

print(loaded_dataset)


DatasetDict({
    train: Dataset({
        features: ['image'],
        num_rows: 120004
    })
})


In [23]:
import torch
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import Dataset, DataLoader
from PIL import Image

In [24]:
# Define a transformation
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to match CNN input size
    transforms.Grayscale(num_output_channels=3),  # Convert to 3-channel grayscale
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Custom Dataset class
class DeepFakeDataset(Dataset):
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform
        self.start_idx = 4  # Ignore the first 4 rows

    def __len__(self):
        return len(self.dataset['train']) - self.start_idx  # Adjust length

    def __getitem__(self, idx):
        idx += self.start_idx  # Shift index to ignore first 4 rows
        image = self.dataset['train'][idx]['image']  # Assuming it's already a PIL.Image
        label = 0 if idx < 90000 else 1  # First 90k are fake, last 30k are real

        if self.transform:
            image = self.transform(image)
        
        return image, torch.tensor(label, dtype=torch.long)


# Create dataset & DataLoader
full_dataset = DeepFakeDataset(loaded_dataset, transform=transform)
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [25]:
import torch.nn as nn
import torch.optim as optim

# Load a pre-trained ResNet model
model = models.resnet18(pretrained=True)

# Modify the classifier layer for binary classification
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)  # Binary classification (fake vs real)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Loss function & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

print("Training complete.")


C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\PIL\TiffImagePlugin.py:900: UserWarning: Co

Epoch [1/10], Loss: 0.3829
Epoch [2/10], Loss: 0.2746
Epoch [3/10], Loss: 0.2159
Epoch [4/10], Loss: 0.1709
Epoch [5/10], Loss: 0.1385
Epoch [6/10], Loss: 0.1156
Epoch [7/10], Loss: 0.0971
Epoch [8/10], Loss: 0.0863
Epoch [9/10], Loss: 0.0757
Epoch [10/10], Loss: 0.0684
Training complete.


In [28]:
from sklearn.metrics import roc_curve
import numpy as np

def compute_eer(y_true, y_scores):
    """
    Calculate Equal Error Rate (EER)
    :param y_true: True labels (1 for real, 0 for fake)
    :param y_scores: Model scores (probabilities or decision values)
    :return: EER value, threshold where EER occurs
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    fnr = 1 - tpr  # False Negative Rate (FRR)
    
    # Find the threshold where FPR ≈ FNR
    eer_threshold = thresholds[np.nanargmin(np.abs(fpr - fnr))]
    eer = fpr[np.nanargmin(np.abs(fpr - fnr))]
    
    return eer, eer_threshold

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score


model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        probabilities = torch.softmax(outputs, dim=1)[:, 1]  
        all_preds.extend(probabilities.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# ---- Compute Metrics ----
accuracy = accuracy_score(all_labels, all_preds > 0.5)
auc = roc_auc_score(all_labels, all_preds)
eer_value, eer_threshold = compute_eer(all_labels, all_preds)

# ---- Print Results ----
print(f"Accuracy: {accuracy:.4f}")
print(f"AUC: {auc:.4f}")
print(f"EER: {eer_value:.4f} (Threshold: {eer_threshold:.4f})")


Accuracy: 0.9020
AUC: 0.9598
EER: 0.1162 (Threshold: 0.0295)
